In [6]:
"""pip install -r requirements.txt"""


'pip install -r requirements.txt'

This file first runs all the tests and than the main function, the purpose is to get a step-by-step walk through of the centire code base, where the results are interactivly achived. It does the same thing as main.py, but used for debugging/education.

Run this from the terminal (while in the project folder):

.venv/bin/python -m pytest tests/ -v


In [7]:
# conftest.py includes pytest fixtures, which won't auto-inject in a notebook
# --> hey are created manually
from pathlib import Path
BASE_DIR = Path()
STAFF_DIR = BASE_DIR / "staff_configs"
DEPT_FILE = BASE_DIR / "department_requirements.txt"
LAW_FILE = BASE_DIR / "swedish_work_law.txt"

In [8]:
# Parse the inputs using config_parser (no LLM functionality)
from src import config_parser
from datetime import date
import dataclasses
import pprint

parsed_inputs = config_parser.parse_all_inputs(
    staff_config_dir = STAFF_DIR,
    dept_req_file = DEPT_FILE,
    law_file = LAW_FILE,
    start_date = date(2026, 4, 20),
    use_llm = False,                                # For now it has to be False as no Anthropic account has been created
)

pprint.pprint(dataclasses.asdict(parsed_inputs))

{'dept_requirements': {'department_name': 'Medicin Avdelning 3 (Med-3)',
                       'min_doctors_day': 1,
                       'min_doctors_evening': 0,
                       'min_doctors_night': 0,
                       'min_doctors_weekend_day': 1,
                       'min_doctors_weekend_evening': 0,
                       'min_doctors_weekend_night': 0,
                       'min_nurses_day': 2,
                       'min_nurses_evening': 1,
                       'min_nurses_night': 1,
                       'min_nurses_weekend_day': 2,
                       'min_nurses_weekend_evening': 1,
                       'min_nurses_weekend_night': 1,
                       'public_holidays': []},
 'law_rules': {'max_consecutive_work_days': 5,
               'max_night_shifts_per_4weeks': 8,
               'max_overtime_per_month_hours': 50,
               'max_overtime_per_year_hours': 200,
               'max_shifts_per_week': 5,
               'min_daily_rest_hour

In [9]:
from src import solver

schedule = solver.build_schedule(
    inputs=parsed_inputs,
    start_date = date(2026, 4, 20),
    num_weeks = 2,
    solver_time_limit_s = 60*5,
)


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 300 log_search_progress: true num_search_workers: 4

Initial optimization model '': (model_fingerprint: 0xb0061051a7c29938)
#Variables: 1'248 (#bools: 648 #ints: 24 in objective) (808 primary variables)
  - 1'224 Booleans in [0,1]
  - 24 in [0,7]
#kAtMostOne: 168 (#literals: 504)
#kBoolAnd: 144 (#enforced: 144) (#literals: 576)
#kBoolOr: 144 (#enforced: 144) (#literals: 432)
#kLinear1: 108
#kLinear2: 612
#kLinearN: 708 (#terms: 5'544)

Starting presolve at 0.00s
  2.77e-03s  0.00e+00d  [DetectDominanceRelations] 
  1.66e-03s  0.00e+00d  [DetectDominanceRelations] 
  2.66e-02s  0.00e+00d  [PresolveToFixPoint] #num_loops=9 #num_dual_strengthening=6 
  6.56e-05s  0.00e+00d  [ExtractEncodingFromLinear] #potential_supersets=485 
  2.58e-04s  0.00e+00d  [DetectDuplicateColumns] 
  6.00e-04s  0.00e+00d  [DetectDuplicateConstraints] #duplicates=24 
[Symmetry] Graph for symmetry has 3'857 nodes and 8'190 arcs.
[Symmetry] Symmet

In [10]:
from src import exporters
exporters.export_excel(
    schedule = schedule,
    inputs = parsed_inputs,
    output_dir = BASE_DIR
)

[exporters] Written schedule_overview.xlsx
